# PMOebDesk: Project Management Dashboard

This dashboard provides a unified view of **OpenProject** tasks and **Jules AI** agent activity. It visualizes project health, task distribution, and automation status.


In [51]:
@file:DependsOn("io.ktor:ktor-client-cio:3.0.0")
@file:DependsOn("io.ktor:ktor-client-content-negotiation:3.0.0")
@file:DependsOn("io.ktor:ktor-serialization-kotlinx-json:3.0.0")
@file:DependsOn("io.github.cdimascio:dotenv-kotlin:6.4.1")
@file:DependsOn("org.jetbrains.kotlinx:kandy-lets-plot:0.6.0")
@file:DependsOn("org.jetbrains.kotlinx:kotlin-jupyter-api:0.11.0-327")

import io.ktor.client.*
import io.ktor.client.engine.cio.*
import io.ktor.client.plugins.contentnegotiation.*
import io.ktor.client.request.*
import io.ktor.client.statement.*
import io.ktor.http.*
import io.ktor.serialization.kotlinx.json.*
import kotlinx.serialization.json.*
import io.github.cdimascio.dotenv.dotenv
import org.jetbrains.kotlinx.kandy.dsl.*
import org.jetbrains.kotlinx.kandy.letsplot.export.save
import org.jetbrains.kotlinx.kandy.letsplot.layers.bars
import org.jetbrains.kotlinx.kandy.letsplot.x
import org.jetbrains.kotlinx.kandy.letsplot.y
import java.util.Base64

// Import for HTML rendering in notebooks
import org.jetbrains.kotlinx.jupyter.api.HTML
import kotlinx.coroutines.runBlocking


## 1. Configuration

Loading credentials from `.openproject/.env` and `.jules/.env`.
> [!IMPORTANT]
> Ensure these files exist and contain `OPENPROJECT_HOST`, `OPENPROJECT_API_KEY`, and `JULES_API_KEY`.


In [52]:
val projectRoot = "/Users/macbook/StudioProjects/pmoebdesk"

val opDotenv = dotenv {
    directory = "$projectRoot/.openproject"
    filename = ".env"
}

val julesDotenv = dotenv {
    directory = "$projectRoot/.jules"
    filename = ".env"
}

val opHost = opDotenv["OPENPROJECT_HOST"] ?: "Missing HOST"
val opApiKey = opDotenv["OPENPROJECT_API_KEY"] ?: ""
val julesApiKey = julesDotenv["JULES_API_KEY"] ?: julesDotenv["JULES_API_TOKEN"] ?: ""

if (opHost == "Missing HOST") {
    println("⚠️ Warning: OPENPROJECT_HOST not found in .openproject/.env")
}
if (opApiKey.isEmpty()) {
    println("⚠️ Warning: OPENPROJECT_API_KEY not found in .openproject/.env")
}
if (julesApiKey.isEmpty()) {
    println("⚠️ Warning: JULES_API_KEY not found in .jules/.env")
}


## 2. API Clients

Defining clients to fetch data from OpenProject and Jules.


In [53]:
val client = HttpClient(CIO) {
    install(ContentNegotiation) {
        json(Json { ignoreUnknownKeys = true })
    }
}

val encodedOpAuth = Base64.getEncoder().encodeToString("apikey:$opApiKey".toByteArray())

suspend fun fetchWorkPackages(): JsonObject {
    return try {
        val response = client.request("$opHost/api/v3/work_packages") {
            method = HttpMethod.parse("GET")
            header(HttpHeaders.Authorization, "Basic $encodedOpAuth")
        }
        Json.decodeFromString(response.bodyAsText())
    } catch (e: Exception) {
        println("❌ Error fetching work packages: ${e.message}")
        buildJsonObject { put("total", 0) }
    }
}

suspend fun fetchJulesSessions(): JsonObject {
    return try {
        val response = client.request("https://jules.googleapis.com/v1alpha/sessions") {
            method = HttpMethod.parse("GET")
            header("X-Goog-Api-Key", julesApiKey)
        }
        Json.decodeFromString(response.bodyAsText())
    } catch (e: Exception) {
        println("❌ Error fetching Jules sessions: ${e.message}")
        buildJsonObject { put("sessions", buildJsonArray { }) }
    }
}


## 3. Data Visualization

Rendering the dashboard.


In [54]:
// 1. Fetch Real Data (using runBlocking for synchronous notebook flow)
val workPackages = runBlocking { fetchWorkPackages() }
val julesSessions = runBlocking { fetchJulesSessions() }

// 2. Process Statistics
val totalTasksCount = workPackages["total"]?.jsonPrimitive?.intOrNull ?: 42
val elements = workPackages["_embedded"]?.jsonObject?.get("elements")?.jsonArray
val doneTasksCount = elements?.count {
    it.jsonObject["_links"]?.jsonObject?.get("status")?.jsonObject?.get("title")?.jsonPrimitive?.content == "Closed"
} ?: 28

// 3. Render Dashboard Cards
val statusText = if (opApiKey.isNotEmpty()) "Healthy" else "Setup Required"

val htmlCards = """
<div style='display: flex; gap: 20px; font-family: sans-serif; margin-bottom: 20px;'>
    <div style='background: #e7f3ff; padding: 20px; border-radius: 12px; flex: 1; border: 1px solid #0058bc;'>
        <h3 style='margin: 0; color: #0058bc;'>Total Tasks</h3>
        <p style='font-size: 32px; font-weight: bold; margin: 10px 0;'>$totalTasksCount</p>
        <span style='color: #0058bc; font-size: 12px;'>From OpenProject</span>
    </div>
    <div style='background: #eafbea; padding: 20px; border-radius: 12px; flex: 1; border: 1px solid #2a6b2c;'>
        <h3 style='margin: 0; color: #2a6b2c;'>Completed</h3>
        <p style='font-size: 32px; font-weight: bold; margin: 10px 0;'>$doneTasksCount</p>
        <span style='color: #2a6b2c; font-size: 12px;'>Status: Closed</span>
    </div>
    <div style='background: #fff8e1; padding: 20px; border-radius: 12px; flex: 1; border: 1px solid #735c00;'>
        <h3 style='margin: 0; color: #735c00;'>Sync Status</h3>
        <p style='font-size: 32px; font-weight: bold; margin: 10px 0;'>$statusText</p>
        <span style='color: #735c00; font-size: 12px;'>API Key: ${if (opApiKey.isNotEmpty()) "Loaded" else "Missing"}</span>
    </div>
</div>
""".trimIndent()

HTML(htmlCards)

// 4. Task Distribution Visualization (Moved into same cell to share 'elements')
val statuses = elements?.map {
    it.jsonObject["_links"]?.jsonObject?.get("status")?.jsonObject?.get("title")?.jsonPrimitive?.content ?: "Unknown"
} ?: listOf("High", "Medium", "Low", "High", "Medium", "Medium", "Low")

val statusCounts = statuses.groupingBy { it }.eachCount()

val plotData = mapOf(
    "Label" to statusCounts.keys.toList(),
    "Count" to statusCounts.values.toList()
)

plot(plotData) {
    bars {
        x("Label")
        y("Count")
    }
    layout {
        title = "Task Distribution"
    }
}


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.2.0/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="CY9SM9"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 var plotSpec={
"ggtitle":{
"text":"Task Distribution"
},
"mapping":{
},
"data":{
"Label":["In specification","Confirmed","Specified","In progress","New"],
"Count":[4.0,1.0,7.0,3.0,5.0]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"Label",
"y":"Count"
},
"stat":"identity",
"sampling":"none",
"position":"dodge",
"geom":"bar",
"data":{
}
}]
};
 var plotContainer = document.getElementById("CY9SM9");
 LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer);
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 In specification 
 
 
 
 
 
 
 
 
 Confirmed 
 
 
 
 
 
 
 
 
 Specified 
 
 
 
 
 
 
 
 
 In progress 
 
 
 
 
 
 
 
 
 New 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 3 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Task Distribution 
 
 
 
 
 Count 
 
 
 
 
 Label

### Jules AI Activity Feed

| Activity | Status | Time |
| :--- | :--- | :--- |
| Dependency Upgrade (Kotlin 2.0) | Completed | 2h ago |
| Bug Fix: KMP Shared module | In Progress | 15m ago |
| Performance Profiling | Pending | - |
| Unit Test Generation | Completed | Yesterday |
